https://judge.nitro-ai.org/competitions/roai-2025/onia/1/view?participant_name=ecaterina

# setup + imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

In [2]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer

In [3]:
train_data = pd.read_csv("/content/train_data.csv")
test_data = pd.read_csv("/content/test_data.csv")

X = train_data["text"]
y = train_data["label"]

In [4]:
subtask1_mask = test_data['subtaskID'] == 1
subtask2_mask = test_data['subtaskID'] == 2

# subtask 1

## data exploration

In [18]:
print("1 (AI): ", len(train_data.loc[train_data["label"]==1, "label"]))
print("0 (om): ", len(train_data.loc[train_data["label"]==0, "label"]))
print(len(train_data))

1 (AI):  3015
0 (om):  6985
10000


## validare

In [19]:
def evaluate(vectorizer, model, X, y):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

  scores = []
  for fit_rows, val_rows in kf.split(X, y):
    X_fit = X.iloc[fit_rows]
    y_fit = y.iloc[fit_rows]
    X_val = X.iloc[val_rows]
    y_val = y.iloc[val_rows]

    X_fit = vectorizer.fit_transform(X_fit)
    X_val = vectorizer.transform(X_val)

    model.fit(X_fit, y_fit)
    preds = model.predict(X_val)
    score = f1_score(y_val, preds, average="macro")

    scores.append(score)
  return np.mean(scores)


## model

In [20]:

options = {
    'words': dict(ngram_range=(1, 2), sublinear_tf=True),
    'words cased': dict(ngram_range=(1, 2), sublinear_tf=True, lowercase=False),
    'chars': dict(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True),
    'chars cased': dict(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, lowercase=False),
}

'''
results = {}
for name, params in options.items():
  vectorizer = TfidfVectorizer(**params)
  model = LogisticRegression()
  results[name] = evaluate(vectorizer, model, X, y)

print(results)
'''

'\nresults = {}\nfor name, params in options.items():\n  vectorizer = TfidfVectorizer(**params)\n  model = LogisticRegression()\n  results[name] = evaluate(vectorizer, model, X, y)\n\nprint(results)\n'

In [21]:
'''bestname = ""
bestr = 0
for name in results.keys():
  r = results[name]
  if r > bestr:
    bestr = r
    bestname = name

print(bestname, bestr)
'''

'bestname = ""\nbestr = 0\nfor name in results.keys():\n  r = results[name]\n  if r > bestr:\n    bestr = r\n    bestname = name\n\nprint(bestname, bestr)\n'

In [22]:
bestname = 'chars'

In [23]:
'''
for C in [1, 10, 100]:
  print('C =', C, evaluate(TfidfVectorizer(**options[bestname]), LogisticRegression(C=C, max_iter=2000), X, y))
'''

"\nfor C in [1, 10, 100]:\n  print('C =', C, evaluate(TfidfVectorizer(**options[bestname]), LogisticRegression(C=C, max_iter=2000), X, y))\n"

In [24]:
vectorizer = TfidfVectorizer(**options[bestname])
model = LogisticRegression(C=10, max_iter=2000)

In [25]:
X_full = vectorizer.fit_transform(X)
model.fit(X_full, y)
preds = model.predict(vectorizer.transform(test_data["text"]))

KeyboardInterrupt: 

# subtask 2


In [13]:
texte = test_data.loc[subtask2_mask, 'text']

vectorizer = TfidfVectorizer(min_df=2, max_df=0.5, sublinear_tf=True)
texvec = vectorizer.fit_transform(texte)

svd = TruncatedSVD(n_components=100, random_state=0)
textsvd = svd.fit_transform(texvec)
textsvd = Normalizer().fit_transform(textsvd)

km = KMeans(n_clusters=4, n_init=10, random_state=0)
km.fit(textsvd)

print(pd.Series(km.labels_).value_counts())

3    206
1    202
2    197
0    195
Name: count, dtype: int64


In [14]:
centers = svd.inverse_transform(km.cluster_centers_)
words = vectorizer.get_feature_names_out()
for c in range(4):
  top = np.argsort(-centers[c])[:15]
  print(c, [words[i] for i in top])

0 ['scientists', 'earth', 'nasa', 'space', 'science', 'researchers', 'planet', 'research', 'new', 'study', 'climate', 'you', 'its', 'solar', 'how']
1 ['god', 'church', 'religious', 'our', 'my', 'jesus', 'you', 'christian', 'faith', 'us', 'me', 'do', 'white', 'love', 'world']
2 ['police', 'she', 'her', 'told', 'shooting', 'according', 'county', 'old', 'him', 'authorities', 'school', 'news', 'suspect', 'charges', 'arrested']
3 ['company', 'its', 'percent', 'business', 'you', 'companies', 'new', 'your', 'employees', 'uber', 'million', 'work', 'ceo', 'like', 'billion']


In [15]:
cluster_to_topic = {0: 'SCIENCE', 1: 'RELIGION', 2: 'CRIME', 3: 'BUSINESS'}

preds2 = []
for c in km.labels_:
  preds2.append(cluster_to_topic[c])

# submisie

In [16]:
subtask1 = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': test_data.loc[subtask1_mask, 'ID'],
    'answer': preds[subtask1_mask]
})

subtask2 = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_data.loc[subtask2_mask, 'ID'],
    'answer': preds2
})

In [17]:
submission = pd.concat([subtask1, subtask2], ignore_index=True)
submission.to_csv('submission.csv', index=False)